# Notebook 3 — The inverse problem: deblurring

Denoising was the case \(G=I\). Here we put a real operator in the middle and
solve the general reconstruction template:
\[
y = Gx + n,\qquad \hat{x}=\argmin_x\;\tfrac12\norm{Gx-y}_2^2 + \lambda R(x).
\]
We use **deblurring** (\(G\) = a blur) because it shows the whole story cleanly:
the naive inverse **explodes** (ill-posedness), and a prior rescues it —
**Tikhonov (L2)** smooths, **total variation (L1)** keeps edges.

> The two medical forward operators — **CT** (\(G\) = Radon transform → FBP) and
> **MRI** (\(G\) = Fourier → k-space) — get their own hands-on notebook in
> `week_3/` (CT & MRI fundamentals). The maths here is identical; only \(G\) changes.

NumPy + Matplotlib only — every operator is built from `np.fft`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2)
plt.rcParams['figure.figsize'] = (10, 4)

def head_phantom(N=96):
    """A simple Shepp-Logan-ish phantom: nested ellipses + a couple of blobs."""
    yy, xx = np.mgrid[0:N, 0:N].astype(float)
    c = (N - 1) / 2.0
    X = (xx - c) / (N / 2.0)
    Y = (yy - c) / (N / 2.0)
    img = np.zeros((N, N))
    img[(X/0.92)**2 + (Y/0.92)**2 <= 1] = 0.3
    img[(X/0.80)**2 + (Y/0.80)**2 <= 1] = 0.6
    img[((X+0.25)/0.18)**2 + ((Y-0.10)/0.18)**2 <= 1] = 0.95
    img[((X-0.30)/0.12)**2 + ((Y+0.20)/0.22)**2 <= 1] = 0.10
    return img

def psnr(ref, test, peak=1.0):
    mse = np.mean((ref - test)**2)
    return 10*np.log10(peak**2 / mse)

clean = head_phantom()
plt.imshow(clean, cmap='gray', vmin=0, vmax=1)
plt.title('phantom (the true image x)'); plt.colorbar(); plt.axis('off'); plt.show()

## Forward model + why the naive inverse is a disaster

Let \(G\) be a Gaussian blur. In the Fourier domain blur is multiplication by an
optical-transfer-function \(K\) that **decays** at high frequencies. The naive
inverse divides by \(K\): \(\hat{x}=\mathcal{F}^{-1}\{Y/K\}\). Dividing by the
near-zero high-frequency values of \(K\) **amplifies the tiny measurement noise
into a catastrophe** — the signature of an ill-posed problem.

In [ ]:
def blur_otf(N, sigma=2.2):
    """OTF K of a Gaussian blur, centred at the origin for circular convolution."""
    yy, xx = np.mgrid[0:N, 0:N]
    cx = np.minimum(xx, N - xx); cy = np.minimum(yy, N - yy)
    psf = np.exp(-(cx**2 + cy**2) / (2*sigma**2))
    psf /= psf.sum()
    return np.fft.fft2(psf)

N = clean.shape[0]
K = blur_otf(N, sigma=2.2)
def G(x):  return np.real(np.fft.ifft2(np.fft.fft2(x) * K))           # forward: blur
def GT(r): return np.real(np.fft.ifft2(np.fft.fft2(r) * np.conj(K)))  # adjoint

blurred = G(clean)
y = blurred + rng.normal(0.0, 0.01, clean.shape)        # tiny 1% noise

naive = np.real(np.fft.ifft2(np.fft.fft2(y) / K))       # divide by K -> blows up

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].imshow(clean, cmap='gray', vmin=0, vmax=1); ax[0].set_title('true x'); ax[0].axis('off')
ax[1].imshow(y, cmap='gray', vmin=0, vmax=1); ax[1].set_title('blurred + 1% noise = y'); ax[1].axis('off')
ax[2].imshow(naive, cmap='gray'); ax[2].set_title('naive inverse Y/K (destroyed)'); ax[2].axis('off')
plt.tight_layout(); plt.show()
print(f'naive-inverse range: [{naive.min():.1f}, {naive.max():.1f}] -- noise amplified wildly')

## Regularise it: Tikhonov (L2) and total variation (L1)

**Tikhonov** has a one-line Fourier solution (Wiener deconvolution):
\(\hat{X}=\dfrac{\overline{K}\,Y}{|K|^2+\lambda}\). The \(+\lambda\) stops the
division from exploding. It works, but smooths edges (it is the Gaussian prior).

**Total variation** keeps edges sharp. No closed form, so we run gradient descent
on \(\tfrac12\norm{Gx-y}^2 + \lambda\,\mathrm{TV}(x)\) with a smoothed TV gradient
\(-\mathrm{div}\!\big(\nabla x/\sqrt{|\nabla x|^2+\varepsilon^2}\big)\).

In [ ]:
def tikhonov_deblur(y, K, lam):
    Y = np.fft.fft2(y)
    Xh = np.conj(K) * Y / (np.abs(K)**2 + lam)
    return np.real(np.fft.ifft2(Xh))

def tv_grad(x, eps=2e-2):
    gx = np.diff(x, axis=1, append=x[:, -1:])
    gy = np.diff(x, axis=0, append=x[-1:, :])
    norm = np.sqrt(gx**2 + gy**2 + eps**2)
    px, py = gx/norm, gy/norm
    div = (px - np.roll(px, 1, axis=1)) + (py - np.roll(py, 1, axis=0))
    return -div

def deblur_tv(y, lam=0.03, iters=300, step=0.18, eps=2e-2):
    x = y.copy()
    for _ in range(iters):
        data_grad = GT(G(x) - y)          # |K|^2 <= 1, so the data term is well-scaled
        x = x - step * (data_grad + lam * tv_grad(x, eps))
    return x

tik = tikhonov_deblur(y, K, lam=0.02)
tv  = deblur_tv(y, lam=0.03)
print(f'Tikhonov PSNR = {psnr(clean, tik):.2f} dB     TV PSNR = {psnr(clean, tv):.2f} dB')

fig, ax = plt.subplots(1, 4, figsize=(15, 4))
for a_, im, t in zip(ax, [y, tik, tv, clean],
                     ['blurred y', 'Tikhonov (L2)', 'total variation (L1)', 'true x']):
    a_.imshow(np.clip(im, 0, 1), cmap='gray', vmin=0, vmax=1); a_.set_title(t); a_.axis('off')
plt.tight_layout(); plt.show()

## Exercises

1. Push the noise to 5% and re-tune Tikhonov's \(\lambda\) and TV's \(\lambda\). Which degrades more gracefully? Compare edge sharpness with a line profile.
2. Implement gradient descent on \(\tfrac12\norm{Gx-y}^2 + \lambda\,\mathrm{TV}(x)\) yourself (you already have `GT(G(x)-y)` and `tv_grad`) and watch the objective decrease.
3. Replace TV with a Huber prior (reuse Notebook 2's `psi_huber`) and compare.
4. Swap the blur \(G\) for a different PSF (motion blur). Does Tikhonov vs TV change?

**Where this goes:** the same template with \(G\)=Radon (CT) and \(G\)=Fourier
(MRI) is in the **`week_3`** notebook. The deep-learning weeks later replace the
hand-built prior \(R(x)\) with a learned one.